Initially we identified the following REITs for our study: ESS, INVH, UDR, MAA, CPT, AMH, WELL, AMT, EQIX and PLD

However, we received other tickers from the query including: AMB ,AMH, AMRS, AMT, CPT, EQIX, ESS, HCN, INVH, MAA, PLD, SCN, UDR, UDRT, WELL

The ticker disrepancies revealed ticker changes, mergers and acquisitions etc. (ProLogis (NYSE: PLD) acquired AMB Property Corporation (NYSE: AMB) in 2011, AmerUs Life Holdings Inc (NYSE: AMRS) by Aviva PLC's in 2006, Welltower Inc. changed its NYSE ticker symbol from HCN to WELL effective at the opening of trading on February 28, 2018)

This inspired to also include Compustat firm controls to account for this as well as different growth stages. Finally, we did manual verification to confirm the correctness of firms and tanges for our study

Data Investigation Summary:

1. Welltower Inc. (formerly HCN) changed its ticker symbol to WELL in February 2018.
    - Unfortunately, there is no Compustat control data till 2018
    - Time Range: HCN 1990-01 to 2018-02, WELL 2018-03 to 2024-12
Decision: Remove since no controls available

2. ESS
    - Compustat data available
    - Time range: 1994-06 to 2024-12
Decision: Keep

3. INVH
    - Compustat data available
    - Time range: 2017-02 to 2024-12
Decision: Remove due to limited time range

4. UDR
    - Compustat data available
    - Time range: 1990-05 to 2024-12
Decision: Keep

5. MAA
    - Compustat data available
    - Time range: 1994-01 to 2024-12
Decision: Keep

6. CPT
    - Compustat data available
    - Time range: 1993-07 to 2024-12 // 1990-01 to 1990-10 belonged to different entity; Missing range: 1990-11 to 1993-06
Decision: Keep

7. AMH
    - Compustat data available
    - Time range: 2013-08 to 2024-12 // 1990-01 to 1997-09 belonged to someone else; Then missing 1997-10 to 1998-01. Then belonging to someone else again 1998-02 to 2006-11; Missing again 2006-12 to 2013-07
Decision: Remove due to limited time range

8. AMT
    - Compustat data available
    - Time range: 1998-06 - 2024-12 // 1990-01 to 1996-07 belonged to someone else; Missing 1996-08 to 1995-05
Decision: Keep (we need more controls)

9. EQUIX
    - Compustat data available
    - Time range: 2000-08 to 2024-12
Decision: Keep (we need more controls)

10. PLD
    - Compustat data available
    - Time range: 1998-07 to 2024-12
Decision: Keep (we need more controls)

Therefore, our final study will include:

Controls --> AMT, EQUIX, PLD

Treatment--> ESS, UDR, MAA, CPT

Time range: 2000-08 - 2024-12

In [5]:
import pandas as pd
import numpy as np
crsp = pd.read_csv('data/wrds_data.csv') 
crsp['date'] = pd.to_datetime(crsp['date'])
crsp['Month'] = crsp['date'].dt.month
crsp['Year'] = crsp['date'].dt.year
crsp['RET'] = pd.to_numeric(crsp['RET'], errors='coerce')
crsp['PRC'] = pd.to_numeric(crsp['PRC'], errors='coerce')
crsp = crsp.dropna(subset=['RET', 'PRC']).copy()
crsp['illiq_daily'] = np.where(
    crsp['VOL'] > 0,
    crsp['RET'].abs() / (crsp['PRC'].abs() * crsp['VOL']),
    np.nan
)
def calculate_monthly_ret(series):
    return (series + 1).prod() - 1
df = crsp.groupby(['TICKER', 'Year', 'Month']).agg({
    'illiq_daily': 'mean',
    'RET': calculate_monthly_ret,    
    'VOL': 'sum'                     
}).reset_index()
df.rename(columns={'RET': 'monthly_ret', 'illiq_daily': 'illiq_avg'}, inplace=True)
df = df.sort_values(['TICKER', 'Year', 'Month']).reset_index(drop=True)
df.columns = df.columns.str.lower()

In [6]:
hcn_months = set(map(tuple, df.loc[df['ticker']=='HCN', ['year','month']].drop_duplicates().values))
well_months = set(map(tuple, df.loc[df['ticker']=='WELL', ['year','month']].drop_duplicates().values))

overlap = sorted(hcn_months & well_months)
print('Overlap months count:', len(overlap))

if overlap:
    dates = [pd.Timestamp(year=int(y), month=int(m), day=1) for y, m in overlap]
    print('First overlap:', dates[0].strftime('%Y-%m'))
    print('Last overlap :', dates[-1].strftime('%Y-%m'))
    print('Sample months:', [d.strftime('%Y-%m') for d in dates[:10]])
else:
    print('No overlapping months found.')

Overlap months count: 63
First overlap: 1993-08
Last overlap : 2018-02
Sample months: ['1993-08', '1993-09', '1993-10', '1993-11', '1993-12', '1994-01', '1994-02', '1994-03', '1994-04', '1994-05']


In [7]:
# they are divergent
temp = df[df['ticker'].isin(['HCN', 'WELL'])]
temp[temp['year'] == 1993]

,ticker,year,month,illiq_avg,monthly_ret,vol
1996,HCN,1993,1,5.130706e-08,0.074854,204100.0
1997,HCN,1993,2,4.370946e-08,-0.000002,293900.0
1998,HCN,1993,3,6.011940e-08,0.077777,266000.0
1999,HCN,1993,4,3.542348e-08,0.035258,315600.0
2000,HCN,1993,5,4.567674e-08,-0.025381,134000.0
2001,HCN,1993,6,5.237431e-08,-0.010415,197500.0
2002,HCN,1993,7,2.041054e-08,0.015268,282400.0
2003,HCN,1993,8,5.712309e-08,0.026457,181600.0
2004,HCN,1993,9,3.329798e-08,0.061856,187600.0
2005,HCN,1993,10,3.086267e-08,-0.019211,1391000.0


In [10]:
temp[temp['year'].isin([2016, 2017, 2018, 2019])].sort_values(['year', 'month']).reset_index(drop=True).head(50)

,ticker,year,month,illiq_avg,monthly_ret,vol
0,HCN,2016,1,8.885125e-11,-0.085403,61652091.0
1,HCN,2016,2,8.796553e-11,0.039039,70761749.0
2,HCN,2016,3,7.216463e-11,0.087173,51032954.0
3,HCN,2016,4,6.820366e-11,0.001154,37134243.0
4,HCN,2016,5,6.568429e-11,0.004256,57059253.0
5,HCN,2016,6,5.620207e-11,0.105355,58142896.0
6,HCN,2016,7,6.043527e-11,0.041487,32743645.0
7,HCN,2016,8,6.159144e-11,-0.021628,39745225.0
8,HCN,2016,9,6.489238e-11,-0.025798,46947184.0
9,HCN,2016,10,7.369109e-11,-0.083456,38014410.0


In [11]:
well = df[df['ticker'] == 'WELL'].copy()
well['date'] = pd.to_datetime(dict(year=well['year'].astype(int), month=well['month'].astype(int), day=1))

if well.empty:
    print('No WELL rows found')
else:
    span = pd.date_range(start=well['date'].min(), end=well['date'].max(), freq='MS')
    present = sorted(pd.to_datetime(well['date'].unique()))
    missing = sorted(set(span) - set(present))

    def months_to_ranges(dates):
        if not dates:
            return []
        dates = sorted(dates)
        ranges = []
        start = prev = dates[0]
        for d in dates[1:]:
            if (d.year * 12 + d.month) == (prev.year * 12 + prev.month) + 1:
                prev = d
            else:
                ranges.append((start, prev))
                start = prev = d
        ranges.append((start, prev))
        return [(s.strftime('%Y-%m'), e.strftime('%Y-%m')) for s, e in ranges]

    print('WELL span:', well['date'].min().strftime('%Y-%m'), 'to', well['date'].max().strftime('%Y-%m'))
    print('Total months in span:', len(span))
    print('Months present     :', len(present))
    print('Missing months     :', len(missing))
    print('First 20 missing   :', [d.strftime('%Y-%m') for d in missing[:20]])
    print('Missing ranges     :', months_to_ranges(missing))

WELL span: 1993-08 to 2024-12
Total months in span: 377
Months present     : 145
Missing months     : 232
First 20 missing   : ['1998-10', '1998-11', '1998-12', '1999-01', '1999-02', '1999-03', '1999-04', '1999-05', '1999-06', '1999-07', '1999-08', '1999-09', '1999-10', '1999-11', '1999-12', '2000-01', '2000-02', '2000-03', '2000-04', '2000-05']
Missing ranges     : [('1998-10', '2018-01')]


In [8]:
def check_ticker_time_coverage(df, ticker, year_col='year', month_col='month', return_presence_df=False):
    """
    Check min/max month and detect gaps for a ticker in a dataframe with year/month columns.
    Returns a dict with:
      - span_start, span_end (YYYY-MM)
      - total_months_in_span (int)
      - present_months_count (int)
      - missing_months_count (int)
      - missing_ranges (list of (YYYY-MM, YYYY-MM))
      - presence_df (optional pandas DataFrame with a row per month in span and `present` flag)
    """
    import pandas as pd

    dfc = df.copy()
    dfc[ticker] = dfc.get('ticker', dfc.get('TICKER', pd.Series())).astype(str).str.upper()  # safe access
    t = dfc[dfc[ticker] == ticker.upper()] if ticker.upper() in dfc[ticker].unique() else dfc[dfc['ticker'].astype(str).str.upper() == ticker.upper()]
    if t.empty:
        return {"error": f"No rows found for ticker {ticker}"}

    # normalize year/month to ints and build month-start dates
    years = t[year_col].astype(int)
    months = t[month_col].astype(int)
    dates = pd.to_datetime(dict(year=years, month=months, day=1))
    dates = dates.drop_duplicates().sort_values()
    start = dates.min()
    end = dates.max()
    span = pd.date_range(start=start, end=end, freq='MS')

    present_set = set(dates)
    missing = sorted(list(set(span) - present_set))

    def _to_ranges(dts):
        if not dts:
            return []
        dts = sorted(dts)
        ranges = []
        s = prev = dts[0]
        for d in dts[1:]:
            if (d.year * 12 + d.month) == (prev.year * 12 + prev.month) + 1:
                prev = d
            else:
                ranges.append((s.strftime('%Y-%m'), prev.strftime('%Y-%m')))
                s = prev = d
        ranges.append((s.strftime('%Y-%m'), prev.strftime('%Y-%m')))
        return ranges

    missing_ranges = _to_ranges(missing)
    result = {
        "span_start": start.strftime('%Y-%m'),
        "span_end": end.strftime('%Y-%m'),
        "total_months_in_span": len(span),
        "present_months_count": len(present_set),
        "missing_months_count": len(missing),
        "missing_ranges": missing_ranges
    }

    if return_presence_df:
        presence_df = pd.DataFrame({"date": span})
        presence_df["present"] = presence_df["date"].isin(present_set)
        presence_df["year"] = presence_df["date"].dt.year
        presence_df["month"] = presence_df["date"].dt.month
        result["presence_df"] = presence_df

    return result

res = check_ticker_time_coverage(df, 'ESS', return_presence_df=True)
print(res['span_start'], 'to', res['span_end'])
print('Missing months:', res['missing_months_count'])
print('Missing ranges sample:', res['missing_ranges'][:10])
res['presence_df'].head()

1994-06 to 2024-12
Missing months: 0
Missing ranges sample: []


,date,present,year,month
0,1994-06-01,True,1994,6
1,1994-07-01,True,1994,7
2,1994-08-01,True,1994,8
3,1994-09-01,True,1994,9
4,1994-10-01,True,1994,10


In [9]:
res = check_ticker_time_coverage(df, 'UDR', return_presence_df=True)
print(res['span_start'], 'to', res['span_end'])
print('Missing months:', res['missing_months_count'])
print('Missing ranges sample:', res['missing_ranges'][:10])
res['presence_df'].head()
# checked that for each

1990-05 to 2024-12
Missing months: 0
Missing ranges sample: []


,date,present,year,month
0,1990-05-01,True,1990,5
1,1990-06-01,True,1990,6
2,1990-07-01,True,1990,7
3,1990-08-01,True,1990,8
4,1990-09-01,True,1990,9
